# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (30-day readmissions) and role-based column masking on sample records using DuckDB.

In [ ]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)
print("DuckDB initialized for Gold Mart and Security simulation.")

In [ ]:
# 1. Setup Mock Silver Table
con.execute("""
    CREATE TABLE condition_occurrence (
        person_id BIGINT,
        condition_concept_id INT,
        condition_start_date DATE,
        ssn VARCHAR,
        dob DATE
    );

    -- 9201 denotes inpatient admission
    INSERT INTO condition_occurrence VALUES 
    (1, 9201, '2023-01-01', '123-45-6789', '1980-05-15'),
    (1, 9201, '2023-01-20', '123-45-6789', '1980-05-15'), -- 19 days later (30-day readmission!)
    (2, 9201, '2023-02-01', '987-65-4321', '1975-10-20'),
    (2, 9201, '2023-04-01', '987-65-4321', '1975-10-20'); -- 59 days later (Not 30-day)
""")
print("Mock silver condition data inserted.")

In [ ]:
# 2. Simulate dbt Gold Model: 30-Day Readmissions
con.execute("""
    CREATE VIEW gold_readmissions AS
    WITH index_admissions AS (
        SELECT person_id, condition_start_date AS index_date, ssn, dob
        FROM condition_occurrence
        WHERE condition_concept_id IN (9201, 262)
    ),
    subsequent_admissions AS (
        SELECT person_id, condition_start_date AS readmission_date
        FROM condition_occurrence
        WHERE condition_concept_id IN (9201, 262)
    )
    SELECT 
        i.person_id,
        i.ssn,
        i.dob,
        i.index_date,
        s.readmission_date,
        CAST(s.readmission_date - i.index_date AS INT) AS days_to_readmission,
        CASE 
            WHEN s.readmission_date - i.index_date <= 30 THEN 1 
            ELSE 0 
        END AS is_30_day_readmission
    FROM index_admissions i
    JOIN subsequent_admissions s 
      ON i.person_id = s.person_id
      AND s.readmission_date > i.index_date
      AND s.readmission_date <= i.index_date + INTERVAL 30 DAY;
""")

print("Gold 30-Day Readmissions Model Output:")
print(con.execute("SELECT * FROM gold_readmissions").df())

In [ ]:
# 3. Simulate Role-Based Column Masking (Dynamic Data Masking)
# DuckDB doesn't have native DDM, so we simulate it with views reflecting role permissions.

# View for EHDIP_DATA_SCIENTIST (No PII access)
con.execute("""
    CREATE VIEW gold_readmissions_scientist_role AS
    SELECT 
        person_id,
        '***-**-****' AS ssn, -- Masked
        DATE_TRUNC('year', dob) AS dob, -- Masked to year
        index_date,
        is_30_day_readmission
    FROM gold_readmissions;
""")

print("\nScientist Role View (Masked PII):")
print(con.execute("SELECT * FROM gold_readmissions_scientist_role").df())